<a href="https://colab.research.google.com/github/bdipesh3045/Data-Science-project-AI-Class-/blob/main/palden/palden_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%pip install -q kagglehub wandb torch scikit-learn matplotlib

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import wandb
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, hinge_loss, ConfusionMatrixDisplay

device = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT = "sign-language-mnist"
print("Using:", device)

Using: cpu


In [11]:
wandb.login()

True

In [12]:
path = kagglehub.dataset_download("datamunge/sign-language-mnist")
train_df = pd.read_csv(glob.glob(os.path.join(path, "**", "sign_mnist_train.csv"), recursive=True)[0])
test_df  = pd.read_csv(glob.glob(os.path.join(path, "**", "sign_mnist_test.csv"),  recursive=True)[0])

X_train = train_df.drop(columns="label").to_numpy(dtype="float32") / 255.0
y_train = train_df["label"].to_numpy()
X_test  = test_df.drop(columns="label").to_numpy(dtype="float32") / 255.0
y_test  = test_df["label"].to_numpy()

labels  = sorted(np.unique(y_train))
letters = [chr(65 + int(l)) for l in labels]
print("Train:", X_train.shape, "| Test:", X_test.shape)

Using Colab cache for faster access to the 'sign-language-mnist' dataset.
Train: (27455, 784) | Test: (7172, 784)


In [13]:
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
print("Batches per epoch:", len(train_loader))

Batches per epoch: 215


In [14]:
def make_model(hidden_layers, activation):
    acts = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}
    layers, in_size = [], 784
    for h in hidden_layers:
        layers += [nn.Linear(in_size, h), acts[activation]()]
        in_size = h
    layers.append(nn.Linear(in_size, 25))   # labels go 0–24
    return nn.Sequential(*layers)

print(make_model([512, 256, 128], "relu"))

Sequential(
  (0): Linear(in_features=784, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ReLU()
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ReLU()
  (6): Linear(in_features=128, out_features=25, bias=True)
)


In [15]:
def run_experiment(name, hidden_layers, activation, lr=1e-3, epochs=20):
    wandb.init(project=PROJECT, name=name,
               config={"model": "MLP", "hidden_layers": hidden_layers,
                       "activation": activation, "learning_rate": lr,
                       "epochs": epochs, "batch_size": 128})

    model   = make_model(hidden_layers, activation).to(device)
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        # Train: the model updates its weights and biases here
        model.train()
        total = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        train_loss = total / len(train_loader.dataset)

        # Evaluate on the test set
        model.eval()
        with torch.no_grad():
            logits    = model(X_test_t.to(device))
            test_loss = loss_fn(logits, y_test_t.to(device)).item()
            test_acc  = (logits.argmax(1).cpu() == y_test_t).float().mean().item()

        wandb.log({"epoch": epoch, "train_loss": train_loss,
                   "test_loss": test_loss, "test_acc": test_acc})

    wandb.finish()
    print(f"{name}: test accuracy {test_acc:.2%}")
    return model, test_acc

In [16]:
experiments = [
    # name                       layers             activation  learning rate
    ("mlp-3layer-relu",          [512, 256, 128],   "relu",     1e-3),   # baseline MLP
    ("mlp-3layer-tanh",          [512, 256, 128],   "tanh",     1e-3),   # activation
    ("mlp-3layer-sigmoid",       [512, 256, 128],   "sigmoid",  1e-3),   # activation
    ("mlp-3layer-relu-small",    [128, 64, 32],     "relu",     1e-3),   # layer size
    ("mlp-3layer-relu-large",    [1024, 512, 256],  "relu",     1e-3),   # layer size
    ("mlp-3layer-relu-lr0.01",   [512, 256, 128],   "relu",     1e-2),   # learning rate
    ("mlp-3layer-relu-lr0.0001", [512, 256, 128],   "relu",     1e-4),   # learning rate
]

results = {}
for name, hidden, act, lr in experiments:
    model, acc = run_experiment(name, hidden, act, lr=lr)
    results[name] = {"model": model, "acc": acc, "hidden": hidden, "act": act, "lr": lr}

print("\nSummary:")
for name, r in sorted(results.items(), key=lambda x: -x[1]["acc"]):
    print(f"  {name:28s} {r['acc']:.2%}")

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▃▄▅▆▅▇▆▇▇▇▇▇█▇██▅██
test_loss,▅▃▂▁▁▃▁▂▂▂▃▃▃▃▅▃▃█▂▃
train_loss,█▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▂▁
epoch,20
test_acc,0.75432
test_loss,1.43389
train_loss,0.00982


mlp-3layer-relu: test accuracy 75.43%


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▃▅▆▇▆▇█████████████
test_loss,█▅▃▂▂▄▁▁▁▁▁▂▂▂▂▂▂▂▂▂
train_loss,█▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
test_acc,0.80703
test_loss,0.79197
train_loss,0.00085


mlp-3layer-tanh: test accuracy 80.70%


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▂▃▄▅▅▅▆▇▆▇▇▇████▇██
test_loss,█▆▅▄▃▃▂▂▁▁▁▂▁▁▁▁▁▂▁▂
train_loss,█▆▅▅▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁
epoch,20
test_acc,0.69004
test_loss,1.25415
train_loss,0.06013


...

mlp-3layer-relu-large: test accuracy 77.40%


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁█▇▅▄▆▄▄▆▇▇▆▅▇▇█▇▄█▆
test_loss,▇▁▁▂▂▂▄▅▃▂▃▄▆▅▅▅▆█▆▇
train_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch,20
test_acc,0.24916
test_loss,2.6766
train_loss,1.62785


mlp-3layer-relu-lr0.01: test accuracy 24.92%


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▃▄▅▅▅▆▅▆▇▇▇▇▇▇█████
test_loss,█▆▅▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁
train_loss,█▆▅▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
epoch,20
test_acc,0.64236
test_loss,1.18633
train_loss,0.55421


mlp-3layer-relu-lr0.0001: test accuracy 64.24%

Summary:
  mlp-3layer-tanh              80.70%
  mlp-3layer-relu-large        77.40%
  mlp-3layer-relu              75.43%
  mlp-3layer-sigmoid           69.00%
  mlp-3layer-relu-lr0.0001     64.24%
  mlp-3layer-relu-small        62.55%
  mlp-3layer-relu-lr0.01       24.92%


In [18]:
wandb.finish()   # close the run that was left open by the error

from sklearn.svm import SVC
from sklearn.metrics import log_loss

In [ ]:
# RBF SVM = Week 1 best model. It doesn't train in epochs,
# so its final loss is drawn as a flat line for comparison.
wandb.init(project=PROJECT, name="svm-rbf-baseline",
           config={"model": "SVM RBF (Week 1 best)", "C": 10})

svm = SVC(kernel="rbf", C=10, gamma="scale", probability=True)   # slow: 10–20 min on CPU
svm.fit(X_train, y_train)

svm_train_loss = log_loss(y_train, svm.predict_proba(X_train), labels=svm.classes_)
svm_test_loss  = log_loss(y_test,  svm.predict_proba(X_test),  labels=svm.classes_)
svm_test_acc   = svm.score(X_test, y_test)

for epoch in range(1, 21):
    wandb.log({"epoch": epoch, "train_loss": svm_train_loss,
               "test_loss": svm_test_loss, "test_acc": svm_test_acc})

wandb.finish()
print(f"RBF SVM: test accuracy {svm_test_acc:.2%}, test loss {svm_test_loss:.3f}")

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
best_name = max(results, key=lambda k: results[k]["acc"])
best = results[best_name]
print("Best model:", best_name)

best["model"].eval()
with torch.no_grad():
    y_pred = best["model"](X_test_t.to(device)).argmax(1).cpu().numpy()

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"Macro F1:      {f1_score(y_test, y_pred, average='macro'):.4f}")

fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=labels,
                                        display_labels=letters, cmap="Blues",
                                        colorbar=False, ax=ax)
ax.set_title(f"Best MLP ({best_name}) — Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
torch.save({"state_dict": best["model"].state_dict(),
            "hidden_layers": best["hidden"],
            "activation": best["act"]}, "best_mlp.pt")

from google.colab import files
files.download("best_mlp.pt")